# 09 Clean S12 holdings sources

目标：清洗Type1/Type3、锁定报告日期并立即将assets重命名为type1_assets。

- 输入：s12type1.csv；s12type3.csv
- 输出：s12_holdings_with_rdate.parquet及冲突/未匹配审计
- 主键：fundno+fdate+cusip8
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：01
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import duckdb
from src.atomic_write import duckdb_copy_csv_atomic, duckdb_copy_parquet_atomic
from src.runtime_paths import output_path

def esc(path): return str(path).replace("'","''")
limit1="" if not SMOKE else " LIMIT 200000"
limit3="" if not SMOKE else " LIMIT 300000"
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=true")
con.execute(f"""CREATE TEMP VIEW type1_raw AS
 SELECT fundno, try_cast(rdate AS DATE) rdate, try_cast(fdate AS DATE) fdate,
        try_cast(assets AS DOUBLE) type1_assets
 FROM (SELECT fundno,rdate,fdate,assets FROM read_csv_auto('{esc(PATHS["s12_type1"])}',header=true,all_varchar=true){limit1})""")
con.execute("""CREATE TEMP TABLE type1 AS
 WITH d AS (SELECT DISTINCT * FROM type1_raw),
 first_fdate AS (
   SELECT *, row_number() OVER(PARTITION BY fundno,rdate ORDER BY fdate) rn FROM d),
 monthly AS (
   SELECT fundno,rdate,fdate,type1_assets,date_trunc('month',rdate)::DATE report_month
   FROM first_fdate WHERE rn=1),
 latest AS (
   SELECT *, row_number() OVER(PARTITION BY fundno,report_month ORDER BY rdate DESC) rn FROM monthly)
 SELECT fundno,rdate,fdate,type1_assets,report_month FROM latest WHERE rn=1""")
con.execute("""CREATE TEMP TABLE conflict_keys AS
 SELECT fundno,fdate,count(DISTINCT rdate) n_rdate FROM type1
 GROUP BY fundno,fdate HAVING count(DISTINCT rdate)>1""")
con.execute(f"""CREATE TEMP VIEW type3 AS
 SELECT DISTINCT fundno,try_cast(fdate AS DATE) fdate,cusip,
        lpad(substr(trim(cusip),1,8),8,'0') cusip8,try_cast(shares AS DOUBLE) shares
 FROM (SELECT fundno,fdate,cusip,shares FROM read_csv_auto('{esc(PATHS["s12_type3"])}',header=true,all_varchar=true){limit3})""")
con.execute("""CREATE TEMP VIEW joined AS
 SELECT h.*,t.rdate,t.type1_assets
 FROM type3 h LEFT JOIN (
   SELECT t.fundno,t.fdate,t.rdate,t.type1_assets FROM type1 t
   LEFT JOIN conflict_keys c USING(fundno,fdate) WHERE c.fundno IS NULL
 ) t USING(fundno,fdate)""")
out=output_path(ROOT,RUN_MODE,"09_clean_s12_holdings_sources","data/interim/09_s12/s12_holdings_with_rdate.parquet")
audit1=output_path(ROOT,RUN_MODE,"09_clean_s12_holdings_sources","data/audit/09_s12_type1_conflicts.csv")
audit2=output_path(ROOT,RUN_MODE,"09_clean_s12_holdings_sources","data/audit/09_s12_unmatched_type3.csv")
duckdb_copy_parquet_atomic(con,"SELECT * FROM joined WHERE rdate IS NOT NULL",out)
duckdb_copy_csv_atomic(con,"SELECT t.*,c.n_rdate,'fundno_fdate_multiple_rdate' reason_code FROM type1 t JOIN conflict_keys c USING(fundno,fdate)",audit1)
duckdb_copy_csv_atomic(con,"SELECT *,'type3_without_type1_report' reason_code FROM joined WHERE rdate IS NULL",audit2)
type1_count=int(con.execute("SELECT count(*) FROM type1").fetchone()[0])
type3_count=int(con.execute("SELECT count(*) FROM type3").fetchone()[0])
clean_count=int(con.execute("SELECT count(*) FROM joined WHERE rdate IS NOT NULL").fetchone()[0])
conflict_count=int(con.execute("SELECT count(*) FROM conflict_keys").fetchone()[0])
unmatched_count=int(con.execute("SELECT count(*) FROM joined WHERE rdate IS NULL").fetchone()[0])
con.execute("SET preserve_insertion_order=false")
dupes=int(con.execute("""SELECT coalesce(sum(n-1),0) FROM (
 SELECT count(*) n FROM read_parquet(?) GROUP BY fundno,fdate,cusip8 HAVING count(*)>1)""",[str(out)]).fetchone()[0])
con.close()
INPUTS_USED=["s12type1.csv","s12type3.csv"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,audit1,audit2]]
INPUT_COUNTS={"type1":type1_count,"type3":type3_count}; OUTPUT_COUNTS={"holdings_with_rdate":clean_count}
PRIMARY_KEY=["fundno","fdate","cusip8"]; DUPLICATE_KEY_COUNT=dupes
EXCLUDED_ROW_COUNT=conflict_count+unmatched_count; EXCEPTION_FILE=str(audit2.relative_to(ROOT)); STAGE_STATUS="passed"; STAGE_NOTES=["No CUSIP aggregation was performed.","Full S12 CSV processing used projected DuckDB scans and external-memory Parquet COPY."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "09_clean_s12_holdings_sources", stage="09", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
